# d × k 实验：串行题 × 约束条数（DA / CoT）

## 目标

在 **task A 的串行指针题**（`current = next[current]`，40 行编号表）上跑完整的 `d × k` 网格：

- `d = 1..10`（串行深度）
- `k = 1..5`（输出约束条数）
- 每格 **40 题**，同一 `d` 下的 `k=1..5` **共用同一批 40 题**
- 两种模式：`DA` / `CoT`，只由 system prompt 决定（模型是非 thinking 版，不传 `enable_thinking`）

核心读出量：**每个格子的联合正确率**（答案对 **且** 所有约束都满足），以及
**热力图 = 每格的 `joint(DA) − joint(CoT)`**。

## 约束怎么放（按你的规定）

| k | 放进 system prompt 的约束 |
|---|---|
| 1 | 只有输出格式（`####ANSWER=<整数>` 必须在最后一行） |
| 2 | 再加 1 条 **C1–C6 里随机抽**的常量行 |
| 3 | 再把**这道题能激活的那一条条件式**加进来（`next_start` 落在它的区间） |
| 4 | 再加 1 条随机约束 |
| 5 | 再加 1 条随机约束 |

**嵌套**：`k=2 ⊂ k=3 ⊂ k=4 ⊂ k=5`，同一道题从 k=2 到 k=5 只是「逐条加」，不换题、不换已有约束。
随机的部分用 `random.Random(SEED|qid)` 固定，重跑可复现。

**k≥4 的随机池**：剩下的 C1–C6 + **未激活的条件式**（它们要求「不要写那一行」，是一种否定约束）。

## 题目筛选

- 只用 `kind == "serial"`、`d ∈ 1..10` 的题。
- 只用 `next_start ∈ 0..35` 的题（36–39 没有任何条件式会激活，k≥3 时会缺一条约束）。
- 每个 `d` 从符合条件的一百来题里按固定种子抽 **40 题**，`k=1..5` 共用。

`next_start` 的定义：`table[initial].value`，等价于 `chains[0].trace[0]`（已在 20 条题上核对过）。

## 产物

| 文件 | 内容 |
|---|---|
| `d_x_k_io.jsonl` | 每条生成一行：`system` / `user` / `output` + 判分 |
| `d_x_k_metrics.json` | 每格汇总、每格违反明细、gap 矩阵、逐题结果 |
| `d_x_k_heatmap.png` | `d × k` 的联合正确率 gap 热力图 |

## 判分口径

- `answer_ok`：`####ANSWER=` 后面的整数 == 标准答案。
- `cons_ok`：格式合法（恰好一个答案行且有整数）+ 答案行是最后一行 + **每条约束都满足**。
- `joint_ok` = `answer_ok and cons_ok`。
- **不惩罚草稿**：多写的推理行（只要在答案行之前）不算违反约束，只记 `n_lines` 作诊断。
- 违反明细记录到「哪一条约束没满足」，包括格式本身（记为 `BASE`）。


In [ ]:
import sys, os, subprocess, importlib.util

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle")

if IN_COLAB or IN_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm", "pandas", "matplotlib"], check=True)
    print("vLLM 安装完成。若提示 restart，请重启会话后从下一格开始。")
else:
    print("非 Colab/Kaggle 环境，跳过安装（本机已装 vLLM 时不需要这一格）。")


# ---- torchaudio / torch 的 CUDA 版本不匹配会导致 from vllm import 直接失败 ----
# 本项目是纯文本任务，用不到 torchaudio；import 失败就卸掉。
# def drop_broken_torchaudio(verbose=True):
#     if importlib.util.find_spec("torchaudio") is None:
#         return False
#     r = subprocess.run([sys.executable, "-c", "import torchaudio"], capture_output=True, text=True)
#     if r.returncode == 0:
#         return False
#     if verbose:
#         tail = (r.stderr or "").strip().splitlines()[-1:] or ["(no stderr)"]
#         print("[fix] torchaudio 导入失败：%s" % tail[0])
#     subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"], check=False)
#     return True


# if drop_broken_torchaudio():
#     print("[fix] 已卸载 torchaudio。若随后 import vllm 仍报同样的错，请重启内核再从头跑。")


## 2. 配置

- 路径从当前目录往上找 `Experiments/`，在仓库任意子目录启动都能跑。
- `DTYPE = "bfloat16"`：4090 支持 bf16，用全精度；**如果换回 T4 必须改成 `"float16"`**（Turing 不支持 bf16）。
- `SMOKE=True` 只跑 `d=1,2 × k=1..5 × 5 题`，用来验证链路。


In [ ]:
import json, os, re, sys, time, random, math
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

# ---------- 路径 ----------
def find_repo():
    here = Path.cwd()
    for p in [here] + list(here.parents):
        if (p / "Experiments" / "data_generation" / "synthesis_pointer_numeric_dataset.jsonl").exists():
            return p
    return here


REPO = find_repo()
DATA_DIR = REPO / "Experiments" / "data_generation"
OUT_DIR = REPO / "Experiments" / "final_experiments"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET = DATA_DIR / "synthesis_pointer_numeric_dataset.jsonl"
CONSTRAINS = OUT_DIR / "constrains.jsonl"

# ---------- 实验网格 ----------
D_GRID = list(range(1, 11))
K_GRID = list(range(1, 6))
N_PER_CELL = 40
SEED = 20261110
MODES = ("DA", "CoT")

# ---------- 模型 ----------
HF_ID = "Qwen/Qwen3-4B-Instruct-2507"
DTYPE = "bfloat16"          # 4090；T4 请改 "float16"
GPU_UTIL = 0.90
MAX_MODEL_LEN = 8192
MAX_NUM_SEQS = 32
TEMPERATURE, TOP_P = 0.6, 0.95
MAX_TOKENS = {"DA": 256, "CoT": 2048}

# ---------- 阶段 ----------
SMOKE = False               # True 时只跑 d=1,2 × 5 题
SMOKE_D = (1, 2)
SMOKE_N = 5
STAGE = "smoke" if SMOKE else "full"

# ---------- 产物 ----------
OUT_IO = OUT_DIR / ("d_x_k_io.jsonl")
OUT_METRICS = OUT_DIR / "d_x_k_metrics.json"
OUT_HEATMAP = OUT_DIR / "d_x_k_heatmap.png"

print("REPO:", REPO)
print("DATASET:", DATASET, DATASET.exists())
print("CONSTRAINS:", CONSTRAINS, CONSTRAINS.exists())
print("grid: d=%s, k=%s, %d questions/cell, modes=%s" % (D_GRID, K_GRID, N_PER_CELL, MODES))
print("stage:", STAGE, "| dtype:", DTYPE)
print("io:", OUT_IO)


## 3. 读题 + 抽样

只保留 `kind == "serial"`、`d ∈ 1..10`、`next_start ∈ 0..35` 的题。
每个 `d` 用固定种子抽 40 题，`k=1..5` 共用这一批。

`next_start` 直接由 `table[initial].value` 算，并和 `chains[0].trace[0]` 对一遍，防止字段理解错。


In [ ]:
serial_all = defaultdict(list)
checked = 0
with open(DATASET, encoding="utf-8") as f:
    for line in f:
        r = json.loads(line)
        if r["kind"] != "serial" or r["d"] not in D_GRID:
            continue
        init = r["chains"][0]["initial"]
        nxt = next(row["value"] for row in r["table"] if row["index"] == init)
        if checked < 20:
            assert nxt == r["chains"][0]["trace"][0], (r["qid"], nxt, r["chains"][0]["trace"])
            checked += 1
        if not (0 <= nxt <= 35):
            continue        # 36-39 没有条件式会激活，k>=3 会缺一条
        serial_all[r["d"]].append({
            "qid": r["qid"],
            "d": r["d"],
            "q": r["q"],
            "gold": int(r["ans"]),
            "start": int(init),
            "next_start": int(nxt),
            "table_size": r["lookup"]["TABLE_SIZE"],
            "ops": r["lookup"]["OPS"][0],
        })

QUESTIONS = []
for d in D_GRID:
    pool = serial_all[d]
    rng = random.Random("%s|d=%d" % (SEED, d))
    picked = rng.sample(pool, min(N_PER_CELL, len(pool)))
    picked.sort(key=lambda x: x["qid"])
    QUESTIONS.extend(picked)
    print("d=%2d: 可用 %3d 题 -> 抽 %2d 题" % (d, len(pool), len(picked)))

assert len(QUESTIONS) == N_PER_CELL * len(D_GRID), len(QUESTIONS)
df_q = pd.DataFrame(QUESTIONS)
print("\nnext_start 范围:", int(df_q["next_start"].min()), "-", int(df_q["next_start"].max()))
print("gold 范围:", int(df_q["gold"].min()), "-", int(df_q["gold"].max()))
display(df_q.head(3)[["qid", "d", "start", "next_start", "gold"]])


## 4. 约束

从 `constrains.jsonl` 读 12 条：6 条条件式（`cls=cond`，依赖 `next_start`）+ 6 条常量（`cls=free`）。

- 常量：要求的行**必须恰好出现一次**。
- 条件式：要求的行「出现一次」当且仅当 `next_start ∈ 区间`；不在区间时写了也算违反。

k≥4 的随机池里，未激活的条件式按同一条规则判——它要求「不要写那一行」。


In [ ]:
CONS = {}
with open(CONSTRAINS, encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        r = json.loads(line)
        CONS[r["id"]] = r

C_IDS = [k for k, v in CONS.items() if v["cls"] == "free"]
COND_IDS = [k for k, v in CONS.items() if v["cls"] == "cond"]
assert len(C_IDS) == 6 and len(COND_IDS) == 6, (C_IDS, COND_IDS)

LITERALS = {k: v["line"] for k, v in CONS.items()}


def line_count(text, literal):
    return sum(1 for ln in str(text).splitlines() if ln.strip() == literal)


def cond_active(cid, next_start):
    return int(next_start) in CONS[cid]["condition"]["values"]


def check_cons(cid, text, next_start):
    """返回这一条约束是否满足。"""
    n = line_count(text, LITERALS[cid])
    if CONS[cid]["cls"] == "free":
        return n == 1
    return (n == 1) == cond_active(cid, next_start)


def active_cond(next_start):
    hits = [c for c in COND_IDS if cond_active(c, next_start)]
    assert len(hits) == 1, (next_start, hits)
    return hits[0]


print("free:", C_IDS)
print("cond:", [(c, CONS[c]["condition"]["values"][0], CONS[c]["condition"]["values"][-1]) for c in COND_IDS])
print("\n示例规则全文（COND_0_5）:\n" + CONS["COND_0_5"]["rule"])


## 5. 约束分配（同一题内嵌套）

```
rng = random.Random(SEED|qid)      # 每题一个，和 k 无关
k=1: []
k=2: [c2]                          # c2 从 C1-C6 抽
k=3: [c2, active]                  # active = 该题 next_start 激活的条件式
k=4: [c2, active, e1]              # e1 从「剩余 C + 未激活条件式」抽
k=5: [c2, active, e1, e2]
```

因此 `k` 是纯粹的「加一条」，同一题、同一批已有约束，只有新加的那条不同。
规则在 prompt 里的顺序就是列表顺序（`R2, R3, ...`）。


In [ ]:
def plan_constraints(q):
    """同一道题，k=1..5 的约束列表（嵌套）。"""
    rng = random.Random("%s|%s" % (SEED, q["qid"]))
    c2 = rng.choice(C_IDS)
    active = active_cond(q["next_start"])
    rest = [c for c in (C_IDS + COND_IDS) if c not in (c2, active)]
    rng.shuffle(rest)
    return {
        1: [],
        2: [c2],
        3: [c2, active],
        4: [c2, active] + rest[:1],
        5: [c2, active] + rest[:2],
    }


PLANS = {q["qid"]: plan_constraints(q) for q in QUESTIONS}

# 自检：嵌套、无重复、k=3 的第三条一定是该题激活的那条
for q in QUESTIONS:
    p = PLANS[q["qid"]]
    for k in range(2, 6):
        assert len(p[k]) == k - 1, (k, p[k])
        assert len(set(p[k])) == len(p[k])
        assert p[k][: len(p[k - 1])] == p[k - 1], "k 不嵌套"
    assert active_cond(q["next_start"]) in p[3]
print("约束分配自检通过 | 示例:")
for q in QUESTIONS[:2]:
    print(" ", q["qid"][:12], "next_start=%2d" % q["next_start"], PLANS[q["qid"]])
cnt = Counter()
for q in QUESTIONS:
    for k in K_GRID:
        for c in PLANS[q["qid"]][k]:
            cnt[c] += 1
print("\n各约束被分配的题次（40 题 × 5 个 k）:", dict(sorted(cnt.items())))


## 6. Prompt 拼装

`prompts.md` 里的骨架是给「单字母答案」写的（`as one letter`）。这个任务的答案是**整数**，
所以把答案行改成 `####ANSWER= followed by the final index as an integer`，CoT 里「entry」也改成「index」。
其余措辞（禁止重复规则、禁止写没要求的行、答案行必须最后）保持不变。


In [ ]:
DA_PREAMBLE = ("Answer with the final answer only: no calculations, no pointer steps, no intermediate "
               "values, no explanation and no comment. Write only the lines that the rules below require, "
               "plus the final answer line. Any other text makes the response invalid.\n")

BASE_RULE = ("Output format. Write the lines required by the rules below first, in the order the rules are "
             "listed, and then write the final answer line. The answer line is always the very last line of "
             "the response: ####ANSWER= followed by the final index as an integer, and nothing may be "
             "written after it. Do not repeat or restate these rules. Do not write any keyword line that "
             "these rules do not require.")

COT_PREAMBLE = ("Solve the problem step by step. Before the final answer line, explicitly write the index "
                "reached after each pointer step, one per line. The keyword lines required by the rules "
                "below must come after those steps and before the final answer line.\n")


def preamble(mode):
    return COT_PREAMBLE if mode == "CoT" else DA_PREAMBLE


def build_system(mode, cids):
    rules = [BASE_RULE] + [CONS[c]["rule"] for c in cids]
    body = "Rules:\n" + "\n".join("R%s. %s" % (j + 1, r) for j, r in enumerate(rules))
    if not cids:
        body += ("\n\nNone of the rules above requires a keyword line, so your response must be the final "
                 "answer line and nothing else.")
    return preamble(mode) + body


def build_messages(mode, q, cids):
    return [
        {"role": "system", "content": build_system(mode, cids)},
        {"role": "user", "content": q["q"]},
    ]


def make_jobs(questions, ks, modes, stage):
    jobs = []
    for mode in modes:
        for q in questions:
            for k in ks:
                cids = PLANS[q["qid"]][k]
                jobs.append({
                    "id": "%s|%s|d%d|k%d|%s" % (stage, mode, q["d"], k, q["qid"]),
                    "stage": stage,
                    "mode": mode,
                    "d": q["d"],
                    "k": k,
                    "qid": q["qid"],
                    "gold": q["gold"],
                    "start": q["start"],
                    "next_start": q["next_start"],
                    "constraints": list(cids),
                    "messages": build_messages(mode, q, cids),
                })
    return jobs


_jobs = make_jobs(QUESTIONS[:1], K_GRID, MODES, "selftest")
assert len(_jobs) == 2 * 5
assert len({j["id"] for j in _jobs}) == len(_jobs)
print("jobs 自检通过 | 示例（DA, k=5）:\n" + "-" * 90)
print([j for j in _jobs if j["mode"] == "DA" and j["k"] == 5][0]["messages"][0]["content"])


## 7. 加载 vLLM（Qwen3-4B-Instruct-2507，bf16）

- `Qwen3-4B-Instruct-2507` 是**非 thinking** 模型，**不要**传 `chat_template_kwargs={"enable_thinking": ...}`。
- 4090 用 `dtype="bfloat16"`；如果哪天回 T4，必须改成 `"float16"`（Turing 不支持 bf16）。
- 采样参数沿用项目统一值：`temperature=0.6`，`top_p=0.95`。


In [ ]:
import torch

# 兜底：torchaudio 版本不匹配会让 from vllm import 直接失败
# import importlib.util as _ilu
# if _ilu.find_spec("torchaudio") is not None:
#     import subprocess as _sp
#     _r = _sp.run([sys.executable, "-c", "import torchaudio"], capture_output=True, text=True)
#     if _r.returncode != 0:
#         print("[fix] torchaudio 不可用，卸载后继续")
#         _sp.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"], check=False)

from vllm import LLM, SamplingParams

if not torch.cuda.is_available():
    raise RuntimeError("没有 CUDA。请在 GPU 机器上执行这一格。")

cap = torch.cuda.get_device_capability(0)
mem_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
if DTYPE == "bfloat16" and cap[0] < 8:
    raise RuntimeError("这张卡不支持 bf16（capability %s），请把 DTYPE 改成 'float16'。" % (cap,))

model_path = HF_ID
llm = LLM(
    model=model_path,
    dtype=DTYPE,
    gpu_memory_utilization=GPU_UTIL,
    max_model_len=MAX_MODEL_LEN,
    max_num_seqs=MAX_NUM_SEQS,
)

CHAT_KW = {}   # 非 thinking 模型：不传 enable_thinking
SP = {m: SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_TOKENS[m]) for m in MODES}
print("GPU: %.1f GiB, capability %s | torch %s cuda %s" % (mem_gib, cap, torch.__version__, torch.version.cuda))
print("SP:", {m: (TEMPERATURE, TOP_P, MAX_TOKENS[m]) for m in MODES})


## 8. 生成与判分

- 结果追加写 `d_x_k_io.jsonl`，按 `id` 断点续跑。
- 每条记录保留 `system` / `user` / `output`，以及这道题的约束列表。
- 判分：`answer_ok`、`cons_ok`、`joint_ok`、逐条 `per_cons`、`n_lines`、`trunc`。

答案行解析：`####ANSWER=` 后面必须是**一个整数**（允许前后空格）。格式不合法、出现多个答案行、
或答案行不是最后一行，都记 `BASE` 违反。


In [ ]:
MARK = "####ANSWER="


def load_done(path):
    if not path.exists():
        return set()
    with open(path, encoding="utf-8") as f:
        return {json.loads(l)["id"] for l in f if l.strip()}


def run_jobs(jobs, path=OUT_IO):
    done = load_done(path)
    todo = [j for j in jobs if j["id"] not in done]
    print("总计 %s，已完成 %s，待跑 %s" % (len(jobs), len(jobs) - len(todo), len(todo)))
    for mode in MODES:
        part = [j for j in todo if j["mode"] == mode]
        batch = 64 if mode == "DA" else 32
        for s in range(0, len(part), batch):
            chunk = part[s:s + batch]
            t0 = time.time()
            outs = llm.chat([j["messages"] for j in chunk], SP[mode], use_tqdm=False, **CHAT_KW)
            dt = time.time() - t0
            with open(path, "a", encoding="utf-8") as f:
                for j, o in zip(chunk, outs):
                    c = o.outputs[0]
                    rec = dict(j)
                    rec.pop("messages", None)
                    rec.update(
                        system=j["messages"][0]["content"],
                        user=j["messages"][1]["content"],
                        output=c.text,
                        prompt_tokens=len(o.prompt_token_ids),
                        n_tokens=len(c.token_ids),
                        finish_reason=c.finish_reason,
                        trunc=(c.finish_reason == "length"),
                        batch_sec=round(dt, 2),
                    )
                    f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            print("[%s] %s/%s  batch %.1fs" % (mode, s + len(chunk), len(part), dt))


def parse_answer(text):
    """返回 (答案整数或 None, 状态说明)。"""
    lines = [l.strip() for l in str(text).splitlines() if l.strip()]
    marks = [i for i, l in enumerate(lines) if l.startswith(MARK)]
    if len(marks) == 0:
        return None, "no_marker"
    if len(marks) > 1:
        return None, "multi_marker"
    i = marks[0]
    if i != len(lines) - 1:
        return None, "not_last"
    raw = lines[i][len(MARK):].strip()
    if not re.fullmatch(r"-?\d+", raw):
        return None, "not_int"
    return int(raw), "ok"


def score_row(rec):
    text = rec["output"]
    ans, why = parse_answer(text)
    base_ok = (why == "ok")
    answer_ok = base_ok and (ans == int(rec["gold"]))

    per = {cid: check_cons(cid, text, rec["next_start"]) for cid in rec["constraints"]}
    cons_ok = base_ok and all(per.values())
    joint_ok = answer_ok and cons_ok

    lines = [l for l in str(text).splitlines() if l.strip()]
    violated = ([] if base_ok else ["BASE"]) + [c for c, ok in per.items() if not ok]
    return {
        "answer": ans,
        "answer_ok": bool(answer_ok),
        "base_ok": bool(base_ok),
        "base_reason": why,
        "cons_ok": bool(cons_ok),
        "joint_ok": bool(joint_ok),
        "per_cons": per,
        "violated": violated,
        "n_lines": len(lines),
        "n_tokens": rec["n_tokens"],
        "trunc": bool(rec["trunc"]),
    }


## 9. 冒烟（可选）

`SMOKE=True` 时只跑 `d=1,2 × k=1..5 × 5 题 × 2 模式` = 100 条，用来确认
① 题干能进模型、② 答案行能解析、③ 约束检查能跑通。跑完把 `SMOKE` 改回 `False` 再跑全量。

这一格在任何情况下都可以跑，不会污染全量（`stage` 不同、id 不同）。


In [ ]:
smoke_qs = [q for q in QUESTIONS if q["d"] in SMOKE_D][:SMOKE_N * len(SMOKE_D)]
smoke_jobs = make_jobs(smoke_qs, K_GRID, MODES, "smoke")
print("smoke questions:", len(smoke_qs), "| jobs:", len(smoke_jobs))
run_jobs(smoke_jobs)

sm = []
if OUT_IO.exists():
    with open(OUT_IO, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rr = json.loads(line)
                if rr["stage"] == "smoke":
                    sm.append(rr)
if sm:
    sdf = pd.DataFrame([{**r, **score_row(r)} for r in sm])
    display(sdf.groupby(["d", "k", "mode"]).agg(
        n=("qid", "size"), answer=("answer_ok", "mean"), cons=("cons_ok", "mean"),
        joint=("joint_ok", "mean"), trunc=("trunc", "mean"), n_lines=("n_lines", "median"),
    ).round(3))
    bad = sdf[~sdf["base_ok"]]
    if len(bad):
        print("\n格式解析失败的样本（前 3 条 base_reason）:", bad["base_reason"].value_counts().to_dict())
        for _, r in bad.head(3).iterrows():
            print("-" * 60, r["d"], r["k"], r["mode"])
            print(repr(r["output"])[:300])


## 10. 全量运行

`d=1..10` × `k=1..5` × 40 题 × 2 模式 = **4000 条**。
可中断；重跑本格会按 `id` 跳过已完成的。

预计 4090 上 20–40 分钟（题干带 40 行表，prefill 占大头）。


In [ ]:
jobs = make_jobs(QUESTIONS, K_GRID, MODES, "full")
print("jobs:", len(jobs))
run_jobs(jobs)

rows = [json.loads(l) for l in open(OUT_IO, encoding="utf-8") if l.strip()]
rows = [r for r in rows if r["stage"] == "full"]
print("full 已完成:", len(rows))
assert len(rows) == len(jobs), "还没跑完，重跑本格续跑"


## 11. 汇总、metrics.json 与热力图

- **每格**（d, k, mode）：`answer_rate` / `cons_rate` / `joint_rate`，以及每格各约束的违反次数。
- **gap 矩阵**：`joint(DA) − joint(CoT)`，就是热力图的取值。
- 逐题结果也写进同一个 `metrics.json`，方便回查「这道题为什么没通过」。

热力图：行 = d（1..10），列 = k（1..5），色标居中于 0（蓝 = CoT 更好，红 = DA 更好）。


In [ ]:
try:
    import matplotlib
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception as e:
    HAVE_MPL = False
    print("[warn] matplotlib 不可用，热力图会退化成 csv:", e)

rows = [json.loads(l) for l in open(OUT_IO, encoding="utf-8") if l.strip()]
rows = [r for r in rows if r["stage"] == "full"]
scored = []
for r in rows:
    s = score_row(r)
    rec = {k: r[k] for k in ("id", "stage", "mode", "d", "k", "qid", "gold", "start", "next_start",
                             "constraints", "finish_reason", "prompt_tokens")}
    rec.update(s)
    rec["output"] = r["output"]
    scored.append(rec)

df = pd.DataFrame(scored)

# ---- 每格汇总 ----
cells = []
for (d, k, mode), g in df.groupby(["d", "k", "mode"]):
    viol = Counter()
    for v in g["violated"]:
        for c in v:
            viol[c] += 1
    cells.append({
        "d": int(d), "k": int(k), "mode": mode, "n": int(len(g)),
        "answer_rate": float(g["answer_ok"].mean()),
        "cons_rate": float(g["cons_ok"].mean()),
        "joint_rate": float(g["joint_ok"].mean()),
        "trunc_rate": float(g["trunc"].mean()),
        "n_lines_median": float(g["n_lines"].median()),
        "violations": dict(sorted(viol.items())),
    })
cell_df = pd.DataFrame(cells)
display(cell_df.pivot_table(index="d", columns=["k", "mode"], values="joint_rate").round(3))

# ---- gap 矩阵 ----
j = df.pivot_table(index=["d", "k"], columns="mode", values="joint_ok", aggfunc="mean").reset_index()
j["gap"] = j["DA"] - j["CoT"]
gap_mat = j.pivot(index="d", columns="k", values="gap").reindex(index=D_GRID, columns=K_GRID)
joint_da = j.pivot(index="d", columns="k", values="DA").reindex(index=D_GRID, columns=K_GRID)
joint_cot = j.pivot(index="d", columns="k", values="CoT").reindex(index=D_GRID, columns=K_GRID)
print("\n联合正确率 gap（DA - CoT）:")
display(gap_mat.round(3))

# ---- 写 metrics.json ----
metrics = {
    "meta": {
        "model": HF_ID,
        "dtype": DTYPE,
        "dataset": str(DATASET.relative_to(REPO)),
        "constrains": str(CONSTRAINS.relative_to(REPO)),
        "stage": "full",
        "d_grid": D_GRID,
        "k_grid": K_GRID,
        "n_per_cell": N_PER_CELL,
        "seed": SEED,
        "modes": list(MODES),
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "max_tokens": MAX_TOKENS,
        "n_rows": int(len(df)),
        "judge": {
            "answer": "####ANSWER=<int> on the last line",
            "cons": "base format + every constraint (nesting k=2..5)",
            "joint": "answer_ok and cons_ok",
            "scratch": "not penalized (only lines after the answer matter)",
        },
    },
    "cells": cells,
    "gap": [
        {"d": int(d), "k": int(k),
         "joint_da": float(joint_da.loc[d, k]), "joint_cot": float(joint_cot.loc[d, k]),
         "gap": float(gap_mat.loc[d, k])}
        for d in D_GRID for k in K_GRID
    ],
    "questions": [
        {k: (bool(v) if isinstance(v, (bool, np.bool_)) else (int(v) if isinstance(v, (int, np.integer)) else v))
         for k, v in rec.items()}
        for rec in scored
    ],
}
with open(OUT_METRICS, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)
print("\nwritten:", OUT_METRICS, "| rows:", len(df))

# ---- 热力图 ----
if HAVE_MPL:
    try:
        fig, ax = plt.subplots(figsize=(7.2, 6.0))
        im = ax.imshow(gap_mat.values.astype(float), cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
        ax.set_xticks(range(len(K_GRID)))
        ax.set_xticklabels(K_GRID)
        ax.set_yticks(range(len(D_GRID)))
        ax.set_yticklabels(D_GRID)
        ax.set_xlabel("k (number of constraints)")
        ax.set_ylabel("d (serial depth)")
        ax.set_title("Joint-accuracy gap (DA - CoT)\nper cell, n=%d" % N_PER_CELL)
        for i in range(len(D_GRID)):
            for jj in range(len(K_GRID)):
                v = gap_mat.values[i, jj]
                if np.isfinite(v):
                    ax.text(jj, i, "%.2f" % v, ha="center", va="center", color="black", fontsize=9)
        fig.colorbar(im, ax=ax, label="DA - CoT")
        fig.tight_layout()
        fig.savefig(OUT_HEATMAP, dpi=150)
        plt.show()
        print("written:", OUT_HEATMAP)
    except Exception as e:
        print("[warn] 画图失败，退化成 csv:", e)
        gap_mat.to_csv(OUT_DIR / "d_x_k_gap_matrix.csv", encoding="utf-8")
        print("written:", OUT_DIR / "d_x_k_gap_matrix.csv")
else:
    gap_mat.to_csv(OUT_DIR / "d_x_k_gap_matrix.csv", encoding="utf-8")
    print("written:", OUT_DIR / "d_x_k_gap_matrix.csv")


## 12. 人工审查（必做）

- 抽 2–3 个格子看原文：`k=1` 时 DA 是不是只写答案；`k=3` 时条件式那一条有没有按题激活。
- 看 `violations`：哪条约束掉得最多？`BASE` 违反多说明答案行格式没稳住。
- 看 gap 的正负：如果大多为负（CoT 更好），说明约束在 DA 下更贵。
- 检查截断率：CoT 有没有被 2048 截断。


In [ ]:
for (d, k) in [(1, 3), (5, 5), (10, 5)]:
    for mode in MODES:
        sub = df[(df["d"] == d) & (df["k"] == k) & (df["mode"] == mode)]
        if not len(sub):
            continue
        print("=" * 100)
        print("d=%d k=%d %s | joint=%.2f cons=%.2f answer=%.2f | constraints=%s" % (
            d, k, mode, sub["joint_ok"].mean(), sub["cons_ok"].mean(), sub["answer_ok"].mean(),
            sub.iloc[0]["constraints"]))
        for _, r in sub.sample(min(2, len(sub)), random_state=SEED).iterrows():
            print("-" * 70, "| violated:", r["violated"])
            print(r["output"][:800])
